# Worksheet 2, Part 2 — Observing log and the radial-velocity campaign
### HE 1002-0755

In [ ]:
import sys, os, json, glob
sys.path.insert(0, "../../../scripts")
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from astropy.io import fits
from astropy.time import Time
from astropy.coordinates import SkyCoord, EarthLocation
import astropy.units as u
from PyAstronomy import pyasl
from mike_multispec import read_multispec, merge_orders

DATA = "../../../data"
EP   = f"{DATA}/he1002-0755_epochs"
os.makedirs("results", exist_ok=True); os.makedirs("plots", exist_ok=True)
LCO = EarthLocation.from_geodetic(lon=-70.69242*u.deg, lat=-29.01423*u.deg, height=2400*u.m)

## 1. Observing log — every file

In [ ]:
files = sorted(glob.glob(f"{EP}/*.fits")) + [
    f"{DATA}/he1002-0755blue_multi_ut090727.fits",
    f"{DATA}/he1002-0755ared_multi_ut090205_07slit.fits"]

rows = []
for f in files:
    h = fits.open(f)[0].header
    orders, _ = read_multispec(f)
    snr = [np.nanmedian(o[2]) for o in orders]
    iso = f"{h['UT-DATE']}T{h['UT-START']}"
    t = Time(iso, scale="utc")
    rows.append(dict(file=os.path.basename(f), arm=h.get("INSTRUME", ""),
                     ut=iso, jd=t.jd, exptime=h.get("EXPTIME"),
                     slit=str(h.get("SLITSIZE")), binning=h.get("BINNING"),
                     airmass=h.get("AIRMASS"), nord=len(orders),
                     snr_max=float(np.nanmax(snr)), snr_med=float(np.nanmedian(snr)),
                     observer=h.get("OBSERVER", "")))
rows.sort(key=lambda r: (r["jd"], r["arm"]))

hdr = f"{'file':<46} {'arm':<10} {'UT start':<20} {'JD':<13} {'exp':>6} {'slit':<10} {'bin':<4} {'airm':>6} {'ord':>4} {'S/Nmax':>7} {'S/Nmed':>7}"
lines = [hdr, "-"*len(hdr)]
for r in rows:
    lines.append(f"{r['file'][:46]:<46} {r['arm']:<10} {r['ut']:<20} {r['jd']:<13.4f} "
                 f"{r['exptime']:>6.0f} {r['slit']:<10} {r['binning']:<4} {str(r['airmass']):>6} "
                 f"{r['nord']:>4} {r['snr_max']:>7.1f} {r['snr_med']:>7.1f}")
txt = "\n".join(lines)
print(txt)
open("results/observing_log.txt", "w").write(txt + "\n")
json.dump(rows, open("results/observing_log.json", "w"), indent=2)
print(f"\n{len(rows)} files, {len({r['ut'] for r in rows})} exposures, "
      f"{len({r['ut'][:10] for r in rows})} nights")
print("observers:", ", ".join(sorted({r["observer"] for r in rows})))

## 2. Duplicates and pairing

In [ ]:
PAIRS = [
 ("he1002-0755ablue_multi_ut090205_07slit.fits", "he1002-0755ared_multi_ut090205_07slit.fits"),
 ("he1002-0755bblue_multi_ut090205_07slit.fits", "he1002-0755bred_multi_ut090205_07slit.fits"),
 ("he1002-0755cblue_multi_ut090205_07slit.fits", "he1002-0755cred_multi_ut090205_07slit.fits"),
 ("he1002-0755_2010_blue_multi.fits",            "he1002-0755_2010_red_multi.fits"),
 ("he1002-0755blue_multi_ut110310_07slit.fits",  "he1002-0755red_multi_ut110310_07slit.fits"),
 ("he1002-0755blue_multi_ut130530_07slit.fits",  "he1002-0755red_multi_ut130530_07slit.fits"),
 ("he1002-0755_2013_may30_blue_multi.fits",      "he1002-0755_2013_may30_red_multi.fits"),
 ("he1002-0755_2013_may31_blue_multi.fits",      "he1002-0755_2013_may31_red_multi.fits"),
 ("he1002-0755_2014_blue_multi.fits",            "he1002-0755_2014_red_multi.fits"),
 ("he1002-0755_2017_blue_multi.fits",            "he1002-0755_2017_red_multi.fits"),
 ("he1002-0755_2018_blue_multi.fits",            "he1002-0755_2018_red_multi.fits"),
]
def path(n):
    p = f"{EP}/{n}"
    return p if os.path.exists(p) else f"{DATA}/{n}"
print(f"{len(PAIRS)} exposures to measure")

## 3. Measuring the velocities

In [ ]:
def load1d(p):
    hh = fits.open(p)[0]; h = hh.header; n = h["NAXIS1"]
    cd = h.get("CD1_1", h.get("CDELT1"))
    return h["CRVAL1"] + cd*(np.arange(n) + 1 - h.get("CRPIX1", 1)), np.asarray(hh.data, float)

w122, f122 = load1d(f"{DATA}/HD122563.fits")
w140, f140 = load1d(f"{DATA}/HD140283.fits")

CAII  = {"Ca II 8498": (8490, 8506), "Ca II 8542": (8534, 8550), "Ca II 8662": (8654, 8670)}
MG    = {"Mg I 4167": (4158, 4176)}
METAL = {"Mg b": (5160, 5200), "Fe 4400-4700": (4400, 4700), "Fe 5250-5500": (5250, 5500),
         "Fe 5500-5800": (5500, 5800), "Fe/Cr 4700-5000": (4700, 5000)}
TELL  = {"H2O 7200": (7160, 7350), "H2O 8200": (8100, 8350), "O2 B-band": (6865, 6935)}
V_TELL_HD140283 = 141.92   # HD 140283 has DOPCOR -142; see .agent/reference.md

def ccf_peak(w, f, wt, ft, lo, hi, vmin, vmax, pad=20):
    m = (w > lo) & (w < hi); mt = (wt > lo - pad) & (wt < hi + pad)
    if m.sum() < 30 or mt.sum() < 30:
        return np.nan
    r, c = pyasl.crosscorrRV(w[m], f[m] - np.nanmedian(f[m]),
                             wt[mt], ft[mt] - np.nanmedian(ft[mt]),
                             vmin, vmax, 0.1, skipedge=10)
    return float(r[np.argmax(c)])

E = []
for b, r in PAIRS:
    orders, hd = [], None
    for fn in (path(b), path(r)):
        o, h = read_multispec(fn)
        if hd is None: hd = h
        orders += o
    orders = [o for o in orders if np.nanmedian(o[2]) >= 3.0]
    nhi = sum(1 for o in orders if np.nanmedian(o[2]) >= 8.0)
    iso = f"{hd['UT-DATE']}T{hd['UT-START']}"
    t = Time(iso, scale="utc", location=LCO)
    sky = SkyCoord(ra=hd["RA-D"]*u.deg, dec=hd["DEC-D"]*u.deg)
    hc = float(sky.radial_velocity_correction("heliocentric", obstime=t).to(u.km/u.s).value)
    rec = dict(file=b, iso=iso, jd=float(t.jd), helcorr=hc, n_snr8=nhi,
               exptime=float(hd["EXPTIME"]), airmass=hd.get("AIRMASS"))
    if not orders:
        rec["flag"] = "no usable orders"; E.append(rec); continue
    w, fl = merge_orders(orders, min_snr=3.0)

    for name, (lo, hi) in {**CAII, **MG}.items():
        rec[name] = ccf_peak(w, fl, w122, f122, lo, hi, 120, 320)
    ca = [rec[k] for k in CAII if np.isfinite(rec[k])]
    rec["ca_mean"]    = float(np.mean(ca)) if ca else np.nan
    rec["ca_scatter"] = float(np.std(ca, ddof=1)) if len(ca) > 1 else np.nan

    mv = [ccf_peak(w, fl, w122, f122, lo, hi, 120, 320) for lo, hi in METAL.values()]
    mv = [v for v in mv if np.isfinite(v)]
    rec["metal_mean"]    = float(np.mean(mv)) if mv else np.nan
    rec["metal_scatter"] = float(np.std(mv, ddof=1)) if len(mv) > 1 else np.nan
    # second template, to catch a CCF that has locked onto a spurious peak
    mv2 = [ccf_peak(w, fl, w140, f140, lo, hi, 120, 320) for lo, hi in METAL.values()]
    mv2 = [v for v in mv2 if np.isfinite(v)]
    rec["metal_mean_hd140283"] = float(np.mean(mv2)) if mv2 else np.nan

    tv = [ccf_peak(w, fl, w140, f140, lo, hi, -260, -30) for lo, hi in TELL.values()]
    tv = [v for v in tv if np.isfinite(v)]
    rec["v_tell"] = float(np.mean(tv)) + V_TELL_HD140283 if tv else np.nan
    E.append(rec)
    print(f"{iso[:10]}  JD {t.jd:.3f}  CaII {rec['ca_mean']:+8.2f} (sc {rec['ca_scatter']:4.2f}) | "
          f"Mg4167 {rec['Mg I 4167']:+8.2f} | metals {rec['metal_mean']:+8.2f} | "
          f"tell {rec['v_tell']:+5.2f} | hc {hc:+7.3f}")

## 4. Quality cuts, and why the Ca II triplet fails

In [ ]:
ok = [e for e in E if np.isfinite(e.get("ca_mean", np.nan))]
x = np.array([e["Mg I 4167"] for e in ok])          # a clean proxy for v_geo
y = np.array([e["ca_mean"] - e["Mg I 4167"] for e in ok])
good = np.isfinite(x) & np.isfinite(y)
r = np.corrcoef(x[good], y[good])[0, 1]
print(f"Ca II - Mg I 4167 offset vs geocentric velocity:  r = {r:.3f}  (n = {good.sum()})")
print("   v_geo spans {:.0f} to {:.0f} km/s; the offset spans {:+.1f} to {:+.1f} km/s".format(
      x[good].min(), x[good].max(), y[good].min(), y[good].max()))
print("\n=> The Ca II triplet is telluric-contaminated here. Not used below.")

## 5. Radial velocity vs Julian date

In [ ]:
use = [e for e in E if e.get("n_snr8", 0) >= 4 and np.isfinite(e.get("Mg I 4167", np.nan))]
# A CCF can lock onto a spurious peak; two independent guards, because at low S/N
# *both* templates can fail the same way (2013-05-31 does exactly that).
bad_tpl = [e for e in E if np.isfinite(e.get("metal_mean", np.nan)) and (
             abs(e["metal_mean"] - e["metal_mean_hd140283"]) > 2.0      # templates disagree
             or abs(e["metal_mean"] - e["Mg I 4167"]) > 3.0             # or disagrees with Mg I
             or e.get("metal_scatter", 0) > 2.0)]                       # or windows disagree
print("rejected (low S/N):   ", [e["iso"][:10] for e in E if e not in use])
print("metal CCF unreliable: ", [e["iso"][:10] for e in bad_tpl], "(Mg I 4167 still OK)")

jd   = np.array([e["jd"] for e in use])
mg   = np.array([e["Mg I 4167"] + e["helcorr"] for e in use])
tell = np.array([e["v_tell"] for e in use])
met  = np.array([(e["metal_mean"] + e["helcorr"])
                 if (np.isfinite(e.get("metal_mean", np.nan)) and e not in bad_tpl) else np.nan
                 for e in use])
mg_c = mg - tell          # remove the epoch-to-epoch wavelength-scale drift

t0 = jd.min(); yr = (jd - t0)/365.25
sig = np.nanstd(mg_c[:3], ddof=1)     # three nights 2 days apart: no orbital change expected
print(f"\nper-epoch sigma from the three 2009 nights = {sig:.2f} km/s")

plt.rcParams.update({
    "font.family": "serif", "font.size": 9,
    "axes.labelsize": 10, "axes.linewidth": 0.9,
    "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.top": True, "ytick.right": True,
    "xtick.minor.visible": True, "ytick.minor.visible": True,
    "legend.frameon": False,
})
C_MG, C_MET, C_TELL = "#000000", "#0072B2", "#D55E00"

fig, ax = plt.subplots(3, 1, figsize=(8.0, 8.2), sharex=True,
                       gridspec_kw=dict(hspace=0.08, height_ratios=[2.1, 2.1, 1.2]))

ax[0].errorbar(jd, mg, yerr=sig, fmt="o", ms=5, color=C_MG, capsize=2, lw=1,
               label="Mg I 4167")
ax[0].errorbar(jd, met, yerr=sig, fmt="s", ms=5, mfc="none", color=C_MET, capsize=2, lw=1,
               label="five metal windows")
ax[0].axhline(np.nanmean(mg), color="0.55", ls=":", lw=1)
ax[0].set_ylabel(r"$v_{\rm helio}$  (km s$^{-1}$)")
ax[0].legend(loc="lower left", fontsize=8.5, ncol=2)
ax[0].set_title("HE 1002-0755: 11 MIKE exposures over 10 nights, 9.1 yr baseline", fontsize=11, pad=8)

ax[1].errorbar(jd, mg_c, yerr=sig, fmt="o", ms=5, color=C_MG, capsize=2, lw=1,
               label="Mg I 4167, telluric-corrected")
s, i = np.polyfit(yr, mg_c, 1)
xx = np.linspace(jd.min(), jd.max(), 50)
ax[1].plot(xx, s*(xx - t0)/365.25 + i, "--", color=C_MET, lw=1.3,
           label=f"linear trend  {s:+.2f} km s$^{{-1}}$ yr$^{{-1}}$")
ax[1].axhline(mg_c.mean(), color="0.55", ls=":", lw=1)
ax[1].set_ylabel(r"$v_{\rm helio}$ corrected  (km s$^{-1}$)")
ax[1].legend(loc="lower left", fontsize=8.5)

ax[2].plot(jd, tell, "o", ms=5, color=C_TELL)
ax[2].axhline(0, color="0.55", ls=":", lw=1)
ax[2].set_ylabel("telluric zero\npoint (km s$^{-1}$)")
ax[2].set_xlabel("Julian date")
for a in ax:
    a.grid(alpha=0.18, lw=0.6)
for ext in ("png", "pdf"):
    fig.savefig(f"plots/rv_vs_jd_HE1002-0755.{ext}", dpi=200, bbox_inches="tight")
plt.show()

from scipy import stats as st
for lab, v in [("raw", mg), ("telluric-corrected", mg_c)]:
    chi2 = float(np.sum((v - v.mean())**2/sig**2)); dof = len(v) - 1
    A = np.vstack([yr, np.ones_like(yr)]).T
    sl = np.linalg.lstsq(A, v, rcond=None)[0][0]
    esl = float(np.sqrt(np.linalg.inv(A.T@A)[0, 0])*sig)
    print(f"\n{lab}:  mean {v.mean():+.2f}  p-p {v.max()-v.min():.2f} km/s")
    print(f"   chi2/dof vs constant = {chi2:.1f}/{dof} = {chi2/dof:.1f}  (p = {st.chi2.sf(chi2, dof):.2g})")
    print(f"   slope = {sl:+.3f} +/- {esl:.3f} km/s/yr  ->  {abs(sl/esl):.1f} sigma")

json.dump(E, open("results/rv_epochs.json", "w"), indent=2, default=float)
print("\nwrote results/rv_epochs.json and plots/rv_vs_jd_HE1002-0755.png")

## 6. Where this leaves the binarity question